In [1]:
import uuid, time
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.writer import DataFrameWriter

In [2]:
layer = "gold"
dataset = "fact_seller_performance"

# Load Spark Session
spark = get_spark_session()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Create logical variables
run_id = str(uuid.uuid4())

In [ ]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:

    writer = DataFrameWriter(spark)
    
    fact_sales = spark.sql("""
        WITH review_score AS (
            SELECT 
                orv.order_id, 
                avg(orv.review_score) as avg_review_score
            FROM olist.silver.order_reviews orv 
            GROUP BY orv.order_id
        ),
        overdue_delivery AS(
            SELECT
                o.order_id,
                COUNT(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 END) AS count_overdue_delivery
            FROM olist.silver.orders o
            GROUP BY o.order_id
        )
        SELECT
            s.seller_id,
            sum(oi.price) AS sum_revenue,
            sum(o.count_overdue_delivery) as count_overdue_delivery,
            avg(orv.avg_review_score) as avg_review_score
        FROM olist.silver.sellers s
        LEFT JOIN olist.silver.order_items oi ON oi.seller_id = s.seller_id
        LEFT JOIN overdue_delivery o ON o.order_id = oi.order_id
        LEFT JOIN review_score orv ON orv.order_id = o.order_id
        GROUP BY s.seller_id
        ORDER BY count_overdue_delivery DESC
    """)
    
    writer.write_delta_batch(fact_sales, "olist", "gold", "fact_seller_performance", mode="overwrite")
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise